# Filestore Tiering — Policy Benchmark Analysis

Grouped bar charts across all presets and policy variants.
Each chart shows one metric; within each preset band the 15 policy bars are colored by policy family.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Load ───────────────────────────────────────────────────────────────────────
CSV_PATH = Path('../bench_results.csv')
df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(df)} rows from {CSV_PATH}')
print(df.columns.tolist())

In [ ]:
# ── Derived columns ────────────────────────────────────────────────────────────
df['hit_pct']        = df['hit_rate'] * 100
df['hot_KB']         = df['bytes_wr_hot']   / 1024
df['cold_KB']        = df['bytes_wr_cold0'] / 1024
df['total_moves_KB'] = df['hot_KB'] + df['cold_KB']
df['reorg_ms']       = df['reorganize_us'] / 1_000
df['ingest_ms']      = df['ingest_us']     / 1_000

# ── Canonical ordering ─────────────────────────────────────────────────────────
PRESETS = ['steady_state', 'frequency_favored', 'recency_favored', 'high_churn', 'hot_set']
PRESET_LABELS = {
    'steady_state':      'Steady\nState',
    'frequency_favored': 'Freq\nFavored',
    'recency_favored':   'Recency\nFavored',
    'high_churn':        'High\nChurn',
    'hot_set':           'Hot\nSet',
}

POLICIES = [
    'basic_lru',
    'arc',
    'lfu',
    'lru_2q',          'lru_2q_small',      'lru_2q_large',
    'lecar',           'lecar_fast',         'lecar_slow',
    'cacheus',         'cacheus_lru_biased', 'cacheus_lfu_biased',
    'decision_tree',   'decision_tree_deep', 'decision_tree_fast',
]

# Policy-family color palette
POLICY_COLORS = {
    'basic_lru':           '#4C72B0',
    'arc':                 '#55A868',
    'lfu':                 '#C44E52',
    # lru-2q family — purples
    'lru_2q':              '#8172B2',
    'lru_2q_small':        '#A89CCC',
    'lru_2q_large':        '#5C4B9B',
    # lecar family — oranges
    'lecar':               '#E07B39',
    'lecar_fast':          '#F5A623',
    'lecar_slow':          '#A0522D',
    # cacheus family — teals
    'cacheus':             '#17BECF',
    'cacheus_lru_biased':  '#2ECC71',
    'cacheus_lfu_biased':  '#0E7F8A',
    # decision-tree family — reds
    'decision_tree':       '#E74C3C',
    'decision_tree_deep':  '#922B21',
    'decision_tree_fast':  '#FF8C94',
}

# Filter to only rows whose policy/preset we expect
df = df[df['preset'].isin(PRESETS) & df['policy'].isin(POLICIES)]

print(f'After filter: {len(df)} rows  |  presets: {df["preset"].nunique()}  |  policies: {df["policy"].nunique()}')

In [ ]:
# ── Shared plotting helper ─────────────────────────────────────────────────────

def grouped_bar(ax, metric, ylabel, title=None, fmt='{:.1f}', scale=1.0, annotate=False):
    """Draw a grouped bar chart: preset bands on x-axis, one bar per policy."""
    n_presets  = len(PRESETS)
    n_policies = len(POLICIES)
    group_width = 0.85
    bar_w = group_width / n_policies

    # Aggregate: mean over runs if RUNS > 1
    agg = df.groupby(['preset', 'policy'])[metric].mean() * scale

    xs = np.arange(n_presets, dtype=float)

    for i, policy in enumerate(POLICIES):
        offset = (i - n_policies / 2 + 0.5) * bar_w
        vals = [agg.get((p, policy), 0.0) for p in PRESETS]
        bars = ax.bar(
            xs + offset, vals, width=bar_w * 0.95,
            color=POLICY_COLORS[policy], alpha=0.88,
            edgecolor='white', linewidth=0.4, label=policy,
        )
        if annotate:
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                            fmt.format(v), ha='center', va='bottom',
                            fontsize=5, rotation=90)

    ax.set_xticks(xs)
    ax.set_xticklabels([PRESET_LABELS[p] for p in PRESETS], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    if title:
        ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(-0.5, n_presets - 0.5)


def policy_legend(fig, ncol=5):
    """Attach a compact legend for all policy colors to the figure."""
    handles = [
        mpatches.Patch(color=POLICY_COLORS[p], label=p)
        for p in POLICIES
    ]
    fig.legend(
        handles=handles, loc='lower center',
        ncol=ncol, fontsize=8.5,
        frameon=True, framealpha=0.9,
        bbox_to_anchor=(0.5, -0.01),
    )

print('Helpers defined.')

## Chart 1 — Hit Rate (%)

**Higher is better.** Fraction of edit operations that landed on a hot (non-symlink) file.
This is the primary quality metric for tiering policy placement.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
grouped_bar(ax, 'hit_pct', 'Hit Rate (%)', title='Hit Rate by Preset and Policy')
policy_legend(fig)
fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig('hit_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 2 — Daemon Move Throughput (Promotions & Demotions)

**Lower is better** for a given hit rate — excess churn means the policy is thrashing.
Stacked bars show promotions (hot writes) and demotions (cold writes) per preset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=False)
grouped_bar(axes[0], 'promotions', 'Count', title='Promotions (cold → hot)')
grouped_bar(axes[1], 'demotions',  'Count', title='Demotions  (hot → cold)')
policy_legend(fig)
fig.suptitle('Daemon Move Throughput by Preset and Policy', fontweight='bold', fontsize=13, y=1.02)
fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig('move_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 3 — Bytes Written to Each Tier (KB)

**Economic I/O cost.** In a cloud deployment (NVMe hot / S3 cold), every byte written
to cold storage is a PUT request charge. Lower cold KB = cheaper policy execution.
Promotions (hot KB) reflect bandwidth cost of pulling data back from cold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=False)
grouped_bar(axes[0], 'hot_KB',  'KB Written', title='Bytes Written → Hot Tier  (Promotions)')
grouped_bar(axes[1], 'cold_KB', 'KB Written', title='Bytes Written → Cold Tier (Demotions)')
policy_legend(fig)
fig.suptitle('Byte Movement by Preset and Policy', fontweight='bold', fontsize=13, y=1.02)
fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig('byte_movement.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 4 — Hit Rate vs. Demotion KB (Efficiency Scatter)

**Upper-left is ideal:** high hit rate, low cold-write cost.
Each point is one (preset, policy) pair. Policies that achieve high hit rate without
excessive demotions are the most cost-efficient.

In [ ]:
agg = df.groupby(['policy', 'preset'])[['hit_pct', 'cold_KB', 'hot_KB']].mean().reset_index()

PRESET_MARKERS = {
    'steady_state':      'o',
    'frequency_favored': 's',
    'recency_favored':   '^',
    'high_churn':        'D',
    'hot_set':           'P',
}

fig, ax = plt.subplots(figsize=(13, 8))

for _, row in agg.iterrows():
    ax.scatter(
        row['cold_KB'], row['hit_pct'],
        color=POLICY_COLORS[row['policy']],
        marker=PRESET_MARKERS[row['preset']],
        s=90, alpha=0.85, edgecolors='white', linewidths=0.5,
        zorder=3,
    )

# Legend: policy colors
policy_handles = [
    mpatches.Patch(color=POLICY_COLORS[p], label=p) for p in POLICIES
]
preset_handles = [
    plt.Line2D([0], [0], marker=m, color='gray', linestyle='None',
               markersize=8, label=p)
    for p, m in PRESET_MARKERS.items()
]

l1 = ax.legend(handles=policy_handles, loc='upper left',
               fontsize=8, title='Policy', title_fontsize=9, ncol=2)
ax.legend(handles=preset_handles, loc='lower right',
          fontsize=9, title='Preset', title_fontsize=9)
ax.add_artist(l1)

ax.set_xlabel('Cold Tier Bytes Written (KB)', fontsize=11)
ax.set_ylabel('Hit Rate (%)', fontsize=11)
ax.set_title('Hit Rate vs. Cold-Write Cost  (upper-left = best)', fontweight='bold', fontsize=12)
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('efficiency_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 5 — Daemon Compute Time (reorganize_ms)

Total time the daemon spent in `reorganize()` during the measurement window.
In production this runs asynchronously (invisible to clients), but high compute
time here indicates algorithmic complexity — relevant when scaling to many files.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
grouped_bar(ax, 'reorg_ms', 'Total reorganize() time (ms)',
            title='Daemon Compute Cost (reorganize) by Preset and Policy')
policy_legend(fig)
fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig('reorg_compute.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 6 — Hit Rate Heatmap (Preset × Policy)

Compact overview: each cell is hit rate %, greener = better.
Useful for spotting which policies consistently win or lose across all presets.

In [ ]:
pivot = (
    df.groupby(['policy', 'preset'])['hit_pct'].mean()
      .unstack('preset')
      .reindex(index=POLICIES, columns=PRESETS)
)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=0)

ax.set_xticks(range(len(PRESETS)))
ax.set_xticklabels([PRESET_LABELS[p] for p in PRESETS], fontsize=10)
ax.set_yticks(range(len(POLICIES)))
ax.set_yticklabels(POLICIES, fontsize=9)

for i, policy in enumerate(POLICIES):
    for j, preset in enumerate(PRESETS):
        val = pivot.loc[policy, preset]
        if not pd.isna(val):
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                    fontsize=8.5,
                    color='black' if val < 60 else 'white')

plt.colorbar(im, ax=ax, label='Hit Rate (%)', shrink=0.8)
ax.set_title('Hit Rate Heatmap (Policy × Preset)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('hit_rate_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Table

In [ ]:
summary = (
    df.groupby(['preset', 'policy'])
      .agg(
          hit_pct    =('hit_pct',    'mean'),
          promotions =('promotions', 'mean'),
          demotions  =('demotions',  'mean'),
          cold_KB    =('cold_KB',    'mean'),
          hot_KB     =('hot_KB',     'mean'),
          reorg_ms   =('reorg_ms',   'mean'),
      )
      .round(2)
)

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
summary